# EHT telescope globe

An interactive map of the 11 telescope sites in the supplied EHT station table. Coordinates are stored in degrees and elevations in metres.

In [1]:
import plotly.graph_objects as go

stations = [
    {"name": "ALMA", "code": "AA", "lat": -23.03217, "lon": -67.75536, "elevation_m": 5043, "country": "Chile", "facility": "Atacama Large Millimeter/submillimeter Array"},
    {"name": "APEX", "code": "AP", "lat": -23.00572, "lon": -67.75928, "elevation_m": 5061, "country": "Chile", "facility": "Atacama Pathfinder Experiment"},
    {"name": "GLT", "code": "GL", "lat": 76.535087, "lon": -68.685908, "elevation_m": 67, "country": "Greenland", "facility": "Greenland Telescope"},
    {"name": "IRAM", "code": "PV", "lat": 37.066145, "lon": -3.392597, "elevation_m": 2864, "country": "Spain", "facility": "IRAM 30-meter telescope"},
    {"name": "JCMT", "code": "JC", "lat": 19.82293, "lon": -155.47709, "elevation_m": 4074, "country": "US", "facility": "James Clerk Maxwell Telescope"},
    {"name": "KP", "code": "KP", "lat": 31.953279, "lon": -111.614806, "elevation_m": 1927, "country": "US", "facility": "ARO 12-meter radio telescope located on Kitt Peak"},
    {"name": "LMT", "code": "LM", "lat": 18.98575, "lon": -97.31481, "elevation_m": 4618, "country": "Mexico", "facility": "Large Millimeter Telescope"},
    {"name": "NOEMA", "code": "PB", "lat": 44.63366, "lon": 5.90669, "elevation_m": 2554, "country": "France", "facility": "Northern Extended Millimeter Array"},
    {"name": "SMA", "code": "SM", "lat": 19.8241, "lon": -155.47823, "elevation_m": 4075, "country": "US", "facility": "Submillimeter Array"},
    {"name": "SMT", "code": "AZ", "lat": 32.7016, "lon": -109.8913, "elevation_m": 3173, "country": "US", "facility": "Submillimeter Telescope"},
    {"name": "SPT", "code": "SP", "lat": -90.0, "lon": 45.0, "elevation_m": 2820, "country": "Antarctica", "facility": "South Pole Telescope"},
]

names = [site["name"] for site in stations]
latitudes = [site["lat"] for site in stations]
longitudes = [site["lon"] for site in stations]
hover_text = [
    f"<b>{site['name']} ({site['code']})</b><br>{site['facility']}<br>{site['country']}<br>Elevation: {site['elevation_m']:,} m<br>({site['lat']:.5f}°, {site['lon']:.5f}°)"
    for site in stations
]

In [2]:
HOME_LAT = 0
HOME_LON = -74

fig = go.Figure(
    go.Scattergeo(
        lon=longitudes,
        lat=latitudes,
        text=hover_text,
        hovertemplate="%{text}<extra></extra>",
        mode="markers",
        marker={"size": 10, "color": "#ff6b35", "line": {"color": "white", "width": 1}},
        name="Telescope site",
    )
)

fig.update_geos(
    projection_type="orthographic",
    projection_scale=0.82,
    projection_rotation=dict(lon=HOME_LON, lat=HOME_LAT, roll=0),
    showland=True, landcolor="#294861",
    showocean=True, oceancolor="#071a2a",
    showcoastlines=True, coastlinecolor="#a9c6d9",
    showlakes=False,
    bgcolor="#0b1020",
    lataxis_showgrid=True, lonaxis_showgrid=True,
)
fig.update_layout(
    title="Event Horizon Telescope: station map",
    height=720,
    margin={"l": 20, "r": 20, "t": 60, "b": 20},
    paper_bgcolor="#0b1020",
    font={"color": "#f5f7fa"},
    updatemenus=[
        {
            "type": "buttons",
            "direction": "left",
            "x": 0.02,
            "y": 0.98,
            "xanchor": "left",
            "yanchor": "top",
            "bgcolor": "#1f3550",
            "bordercolor": "#a9c6d9",
            "font": {"color": "#0b1020"},
            "buttons": [
                {
                    "label": "Reset map",
                    "method": "relayout",
                    "args": [{
                        "geo.projection.rotation.lon": HOME_LON,
                        "geo.projection.rotation.lat": HOME_LAT,
                        "geo.projection.rotation.roll": 0,
                    }],
                }
            ],
        }
    ],
)
fig.show()

In [3]:
assert len(stations) == 11, "The project station table should contain 11 sites."
assert all(-90 <= site["lat"] <= 90 for site in stations)
assert all(-180 <= site["lon"] <= 180 for site in stations)
print(f"Globe ready with {len(stations)} telescope sites.")

Globe ready with 11 telescope sites.


In [5]:
import numpy as np
from datetime import datetime, timezone
from itertools import combinations
import plotly.graph_objects as go

#Choose the source and observing setup

TARGET_NAME = "M87*"

# M87* J2000 / ICRS coordinates
SOURCE_RA_DEG = 187.70593
SOURCE_DEC_DEG = 12.39112

# 230 GHz (1.3mm) wavelength
WAVELENGTH_M = 1.3e-3
MIN_ELEVATION_DEG = 15.0

OBS_TIME_UTC = datetime(2026, 4, 11, 0, 0, 0, tzinfo=timezone.utc)

WGS84_A_M = 6_378_137.0
WGS84_F = 1 / 298.257223563
WGS84_E2 = WGS84_F * (2 - WGS84_F)


def geodetic_to_ecef(latitude_deg, longitude_deg, elevation_m):
    """Convert geodetic latitude, longitude, elevation to ECEF metres."""
    lat = np.radians(latitude_deg)
    lon = np.radians(longitude_deg)

    N = WGS84_A_M / np.sqrt(1 - WGS84_E2 * np.sin(lat) ** 2)

    x = (N + elevation_m) * np.cos(lat) * np.cos(lon)
    y = (N + elevation_m) * np.cos(lat) * np.sin(lon)
    z = (N * (1 - WGS84_E2) + elevation_m) * np.sin(lat)

    return np.array([x, y, z])


def local_up_ecef(latitude_deg, longitude_deg):
    """Return the local zenith direction in the Earth-fixed frame."""
    lat = np.radians(latitude_deg)
    lon = np.radians(longitude_deg)

    return np.array([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat),
    ])


def julian_date(utc_time):
    """Convert a timezone-aware UTC datetime to Julian Date."""
    unix_seconds = utc_time.timestamp()
    return unix_seconds / 86400.0 + 2440587.5


def gmst_radians(utc_time):
    """Approximate Greenwich Mean Sidereal Time in radians."""
    jd = julian_date(utc_time)
    T = (jd - 2451545.0) / 36525.0

    gmst_deg = (
        280.46061837
        + 360.98564736629 * (jd - 2451545.0)
        + 0.000387933 * T**2
        - T**3 / 38710000.0
    )

    return np.radians(gmst_deg % 360.0)


def rotate_ecef_to_eci(vector_ecef, utc_time):
    """
    Rotate an Earth-fixed vector into an Earth-centered inertial approximation.
    This is the Earth-rotation step that makes UV tracks move over time.
    """
    theta = gmst_radians(utc_time)
    rotation_matrix = np.array([
        [np.cos(theta), -np.sin(theta), 0],
        [np.sin(theta),  np.cos(theta), 0],
        [0,              0,             1],
    ])
    return rotation_matrix @ vector_ecef


def source_unit_vector(ra_deg, dec_deg):
    """Unit vector pointing from Earth toward a celestial source."""
    ra = np.radians(ra_deg)
    dec = np.radians(dec_deg)

    return np.array([
        np.cos(dec) * np.cos(ra),
        np.cos(dec) * np.sin(ra),
        np.sin(dec),
    ])


def source_altitude_deg(station_up_ecef, target_vector_eci, utc_time):
    """
    Calculate target altitude at one station.
    Positive altitude means above the geometric horizon.
    """
    station_up_eci = rotate_ecef_to_eci(station_up_ecef, utc_time)
    sine_altitude = np.clip(np.dot(station_up_eci, target_vector_eci), -1, 1)
    return np.degrees(np.arcsin(sine_altitude))




target_vector_eci = source_unit_vector(SOURCE_RA_DEG, SOURCE_DEC_DEG)

station_ecef = [
    geodetic_to_ecef(site["lat"], site["lon"], site["elevation_m"])
    for site in stations
]

station_up_vectors = [
    local_up_ecef(site["lat"], site["lon"])
    for site in stations
]



# Calculate UV coverage 

u_coords = []
v_coords = []
hover_text = []

visible_baselines = set()

# Define east and north axes on the sky around the target.
celestial_north = np.array([0.0, 0.0, 1.0])
u_axis = np.cross(celestial_north, target_vector_eci)
u_axis /= np.linalg.norm(u_axis)
v_axis = np.cross(target_vector_eci, u_axis)
v_axis /= np.linalg.norm(v_axis)

# Calculate visibility once, at the selected instant.
utc_time = OBS_TIME_UTC

altitudes = [
    source_altitude_deg(station_up_vectors[i], target_vector_eci, utc_time)
    for i in range(len(stations))
]

visible = [
    altitude >= MIN_ELEVATION_DEG
    for altitude in altitudes
]

# One UV coordinate per usable baseline at this instant.
for i, j in combinations(range(len(stations)), 2):
    if not (visible[i] and visible[j]):
        continue

    baseline_ecef_m = station_ecef[j] - station_ecef[i]
    baseline_eci_m = rotate_ecef_to_eci(baseline_ecef_m, utc_time)

    u = np.dot(baseline_eci_m, u_axis) / WAVELENGTH_M
    v = np.dot(baseline_eci_m, v_axis) / WAVELENGTH_M

    pair_name = f"{stations[i]['name']}–{stations[j]['name']}"
    visible_baselines.add(pair_name)

    details = (
        f"<b>{pair_name}</b>"
        f"<br>UTC: {utc_time:%Y-%m-%d %H:%M}"
        f"<br>{stations[i]['name']} altitude: {altitudes[i]:.1f}°"
        f"<br>{stations[j]['name']} altitude: {altitudes[j]:.1f}°"
        f"<br>u: {u / 1e9:.2f} Gλ"
        f"<br>v: {v / 1e9:.2f} Gλ"
    )

    # The second point is the standard mirrored/conjugate UV point.
    u_coords.extend([u / 1e9, -u / 1e9])
    v_coords.extend([v / 1e9, -v / 1e9])
    hover_text.extend([details, details + "<br><i>Conjugate point</i>"])
# to plot

uv_fig = go.Figure()

uv_fig.add_trace(
    go.Scattergl(
        x=u_coords,
        y=v_coords,
        mode="markers",
        marker={
            "size": 5,
            "color": "#00d9ff",
            "opacity": 0.72,
        },
        text=hover_text,
        hovertemplate="%{text}<extra></extra>",
        name="Visible baseline samples",
    )
)

uv_fig.add_trace(
    go.Scatter(
        x=[0],
        y=[0],
        mode="markers",
        marker={"size": 8, "color": "#ffffff", "symbol": "x"},
        hovertemplate="UV origin<extra></extra>",
        name="Origin",
    )
)

uv_fig.update_layout(
    title=(
        f"{TARGET_NAME}: Earth-rotation UV coverage "
        f"({len(visible_baselines)} usable baselines)"
    ),
    xaxis_title="u (Gλ)",
    yaxis_title="v (Gλ)",
    height=720,
    paper_bgcolor="#0b1020",
    plot_bgcolor="#0b1020",
    font={"color": "#f5f7fa"},
    hovermode="closest",
    legend={"orientation": "h", "y": 1.02},
)

uv_fig.update_xaxes(
    showgrid=True,
    gridcolor="#263448",
    zeroline=True,
    zerolinecolor="#7f8ca3",
)

uv_fig.update_yaxes(
    showgrid=True,
    gridcolor="#263448",
    zeroline=True,
    zerolinecolor="#7f8ca3",
    scaleanchor="x",
    scaleratio=1,
)

uv_fig.show()




total_possible_baselines = len(stations) * (len(stations) - 1) // 2

print(f"Target: {TARGET_NAME}")
print(f"Instant: {OBS_TIME_UTC:%Y-%m-%d %H:%M UTC}")
print(f"Minimum station elevation: {MIN_ELEVATION_DEG}°")
print(f"Total possible station pairs: {total_possible_baselines}")
print(f"Baselines with at least one visible sample: {len(visible_baselines)}")

Target: M87*
Instant: 2026-04-11 00:00 UTC
Minimum station elevation: 15.0°
Total possible station pairs: 55
Baselines with at least one visible sample: 10
